# Tyre Degradation Analysis
How much slower do laps get as tyres age, and how does this differ between Soft, Medium and Hard compounds?

## 1. Setup

In [ ]:
import os
import fastf1
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("../f1_cache", exist_ok=True)
fastf1.Cache.enable_cache("../f1_cache")

## 2. Load one race
Using the 2025 Italian Grand Prix (Monza) to explore the data before the full season CSVs are ready.

In [ ]:
session = fastf1.get_session(2025, "Monza", "R")
session.load(telemetry=False)
laps = session.laps
print(len(laps), "laps loaded")

## 3. Explore the tyre columns
- `Compound`: SOFT, MEDIUM or HARD
- `TyreLife`: how many laps that set of tyres has done
- `Stint`: which stint the driver is on (1 = first set, 2 = after the first pit stop, ...)

In [ ]:
laps[["Driver", "LapNumber", "LapTime", "Compound", "TyreLife", "Stint", "PitInTime", "TrackStatus"]].head(20)

## 4. Clean the laps
Remove pit in/out laps, safety car and yellow flag laps, unusually slow laps, and laps with no time.

In [ ]:
clean = laps.pick_wo_box()           # remove pit in/out laps
clean = clean.pick_track_status("1") # green flag only (no safety car)
clean = clean.pick_quicklaps()       # remove laps slower than 107% of the fastest
clean = clean[clean["LapTime"].notna()].copy()
clean["LapTimeSeconds"] = clean["LapTime"].dt.total_seconds()

print(len(laps), "->", len(clean), "laps after cleaning")

## 5. Lap time vs tyre age
**Note:** cars also get faster during a race as fuel burns off, which works against tyre wear. Raw plots can therefore understate degradation. This is handled later by including `LapNumber` in the regression.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=clean, x="TyreLife", y="LapTimeSeconds", hue="Compound", alpha=0.6)
plt.title("Lap Time vs Tyre Age - Monza 2025")
plt.xlabel("Tyre age (laps)")
plt.ylabel("Lap time (seconds)")
plt.show()